# Trabalho Prático III: Recomendador de Disciplinas com NLP e Busca Semântica

**Disciplina:** Fundamentos de Inteligência Artificial (FIA) — Engenharia de Sistemas (UFMG)

**Objetivo:** Construir um sistema de recomendação de disciplinas que utiliza **Sentence-BERT** para busca semântica sobre ementas reais, combinado com filtros estruturais de **pré-requisitos**, **balanceamento de dificuldade** e **conflitos de horário**.

---

## Pipeline do Recomendador

1. **Curadoria** — 76 disciplinas com ementas do site GEES UFMG + planilha de dificuldade + grade horária oficial
2. **Embeddings (SBERT)** — `neuralmind/bert-base-portuguese-cased` gera vetores densos (768-d)
3. **Busca Semântica** — Similaridade de Cosseno entre perfil do aluno e ementas
4. **Filtros Estruturais** — Pré-requisitos → Dificuldade → Horários (sem choque de grade)

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import torch
import json
import warnings
warnings.filterwarnings('ignore')

from src.config import *
from src.data_curation import get_discipline_table, get_completed_profile_text, get_discipline_text
from src.sbert_encoder import load_sbert_model, encode_texts, encode_disciplines, semantic_search
from src.recommender_rules import (
    filter_recommended_candidates,
    greedy_balance_by_difficulty,
    get_schedule_slots,
    filter_by_schedule,
)

print("Dependências carregadas.")

In [ ]:
# Carregar tabela de disciplinas curadas
df = get_discipline_table()
print(f"Disciplinas: {len(df)}  |  Com ementa detalhada: {sum(~df['ementa'].str.startswith('Estudo dos fundamentos'))}  |  Com horários: {sum(df['horarios'].apply(lambda h: len(h) if isinstance(h, list) else 0) > 0)}")
display(df[['codigo','nome','periodo_sugerido','dificuldade_geral']].head(8))

In [ ]:
# Carregar modelo SBERT e gerar embeddings do corpus
model = load_sbert_model()
corpus_embeddings = encode_disciplines(model, df)
print(f"Embeddings: {corpus_embeddings.shape}")

---
## Função auxiliar: pipeline completo de recomendação

Encapsula busca semântica + pré-requisitos + dificuldade + horários em uma única chamada.

In [ ]:
def recomendar(interesse: str, concluidas: list[str], *, top_k: int = 20, budget: float = 25.0, max_cursos: int = 5):
    """Pipeline completo: perfil semântico → filtros estruturais → grade sugerida."""
    perfil_texto = get_completed_profile_text(df, concluidas)
    query = f"{perfil_texto} {interesse}"

    query_emb = encode_texts(model, [query], batch_size=1)[0]
    hits = semantic_search(query_emb, corpus_embeddings, top_k=top_k)

    candidatas = []
    for hit in hits:
        row = df.iloc[hit['index']].copy()
        row['predicted_probability'] = hit['score']
        row['difficulty'] = row['dificuldade_geral']
        candidatas.append(row)
    df_cand = pd.DataFrame(candidatas)

    # 1. Remover já concluídas
    df_cand = df_cand[~df_cand['codigo'].isin(concluidas)]

    # 2. Filtrar pré-requisitos
    df_validas = filter_recommended_candidates(df_cand, concluidas)

    # 3. Balancear dificuldade + evitar conflitos de horário
    df_final = greedy_balance_by_difficulty(df_validas, difficulty_budget=budget, max_courses=max_cursos, respect_schedule=True)

    print(f"Query: \"{interesse}\"")
    print(f"Concluídas: {concluidas}")
    print(f"Candidatas (top {top_k}): {len(df_cand)}  |  Pós pré-requisitos: {len(df_validas)}  |  Recomendadas: {len(df_final)}")
    print()
    for _, row in df_final.iterrows():
        slots = get_schedule_slots(row)
        horario_str = ', '.join(f'{d} {t}' for d, t in slots) if slots else 'sem horário'
        print(f"  {row['codigo']} — {row['nome']}")
        print(f"    Score: {row['predicted_probability']:.4f}  |  Dificuldade: {row['difficulty']}  |  Horário: {horario_str}")
    return df_final

---
## Cenário 1 — Aluno no 3º período: recomendação padrão do próximo semestre

Estudante concluiu as obrigatórias até o 2º período. Quer saber o que cursar no 4º período com foco em **computação e software**.

In [ ]:
concluidas_s1 = ['MAT001','MAT038','QUI628','ELE630','DCC217', 'MAT039','FIS065','DCC203','EMT122']
recomendar("Tenho muito interesse em programação, software e algoritmos. Gosto da área de computação.", concluidas_s1, budget=22.0, max_cursos=5)

---
## Cenário 2 — Aluno no 5º período com interesse em IA e Dados

Já passou pela base de programação e matemática. Agora quer disciplinas voltadas a **inteligência artificial, machine learning e ciência de dados**.

In [ ]:
concluidas_s2 = [
    'MAT001','MAT038','QUI628','ELE630','DCC217',  # 1º
    'MAT039','FIS065','DCC203','EMT122',             # 2º
    'MAT002','FIS069','DCC204','DCC205','DCC218',    # 3º
    'MAT015','FIS086','FIS152','ELE064','ELT124','EST773',  # 4º
]
recomendar("Meu foco é inteligência artificial, aprendizado de máquina e análise de dados. Quero trabalhar com ciência de dados.", concluidas_s2, budget=22.0, max_cursos=5)

---
## Cenário 3 — Calouro (1º período): descobrindo o que fazer

Acabou de entrar no curso, não tem pré-requisitos cumpridos. Gosta de **tecnologia, inovação e empreendedorismo**.

In [ ]:
concluidas_s3 = []
recomendar("Sou calouro, gosto de tecnologia, inovação e empreendedorismo. Quero matérias que me introduzam à engenharia.", concluidas_s3, budget=20.0, max_cursos=6)

---
## Cenário 4 — Aluno no 4º período precisa de uma optativa

Já fez as obrigatórias até o 3º período. Tem **horário livre na quinta-feira às 19:00** e quer uma optativa que complemente seu interesse em **sistemas embarcados e hardware**.

In [ ]:
concluidas_s4 = [
    'MAT001','MAT038','QUI628','ELE630','DCC217',
    'MAT039','FIS065','DCC203','EMT122',
    'MAT002','FIS069','DCC204','DCC205','DCC218',
]
df_opt = recomendar("Gosto de hardware, circuitos e sistemas embarcados. Preciso de uma optativa nessa área.", concluidas_s4, budget=15.0, max_cursos=2)

# Destacar apenas optativas
optativas = df_opt[df_opt['natureza'] == 'OP'] if len(df_opt) > 0 else df_opt
print(f"\n→ Optativas sugeridas: {len(optativas)}")
for _, row in optativas.iterrows():
    print(f"   {row['codigo']} — {row['nome']} (Score: {row['predicted_probability']:.4f})")

---
## Cenário 5 — Disciplina complementar: "Já fiz X, o que combina?"

Aluno concluiu **Fundamentos de IA (EEE048)** e quer matérias que **complementem/aprofundem** esse conhecimento, independentemente do período.

In [ ]:
# Simula um aluno que já cursou toda a base + FIA
concluidas_s5 = [
    'MAT001','MAT038','QUI628','ELE630','DCC217','MAT039','FIS065','DCC203','EMT122',
    'MAT002','FIS069','DCC204','DCC205','DCC218','MAT015','FIS086','FIS152','ELE064','ELT124','EST773',
    'ELE065','ELE028','ELT029','ELT084','ELT123','ELE631','ESA019',
    'ELT136','EEE050','ELE082','ELE077',
    'EEE048',  # <- FIA concluída!
]
recomendar("Já fiz Fundamentos de IA. Quero disciplinas que aprofundem em machine learning, deep learning, NLP e visão computacional.", concluidas_s5, budget=20.0, max_cursos=4)

---
## Cenário 6 — Aluno avançado (8º período): montando reta final

Já fez quase tudo. Precisa das últimas obrigatórias + TCC e quer **evitar conflitos de horário** com disciplinas de períodos anteriores ainda pendentes.

In [ ]:
concluidas_s6 = [
    'MAT001','MAT038','QUI628','ELE630','DCC217','MAT039','FIS065','DCC203','EMT122',
    'MAT002','FIS069','DCC204','DCC205','DCC218','MAT015','FIS086','FIS152','ELE064','ELT124','EST773',
    'ELE065','ELE028','ELT029','ELT084','ELT123','ELE631','ESA019',
    'ELT136','EEE050','ELE082','ELE077',
    'ELE632','ELE633','EEE046','EEE017','ELE088',
    'EEE048','EEE049',
]
recomendar("Estou no final do curso. Preciso fechar as matérias que faltam do 8º e 9º períodos.", concluidas_s6, budget=25.0, max_cursos=5)

---
## Comparativo entre Cenários

Resumo do funcionamento dos filtros estruturais (pré-requisitos + dificuldade + horários) em cada cenário.

In [ ]:
cenarios = {
    "1. 3º período — Software":    ("programação, software, algoritmos", ['MAT001','MAT038','QUI628','ELE630','DCC217','MAT039','FIS065','DCC203','EMT122']),
    "2. 5º período — IA e Dados":  ("inteligência artificial, machine learning, dados", ['MAT001','MAT038','QUI628','ELE630','DCC217','MAT039','FIS065','DCC203','EMT122','MAT002','FIS069','DCC204','DCC205','DCC218','MAT015','FIS086','FIS152','ELE064','ELT124','EST773']),
    "3. Calouro — Tecnologia":     ("tecnologia, inovação, empreendedorismo", []),
    "4. Optativa — Hardware":      ("hardware, circuitos, embarcados", ['MAT001','MAT038','QUI628','ELE630','DCC217','MAT039','FIS065','DCC203','EMT122','MAT002','FIS069','DCC204','DCC205','DCC218']),
    "5. Complementar à FIA":       ("machine learning, deep learning, NLP", ['MAT001','MAT038','QUI628','ELE630','DCC217','MAT039','FIS065','DCC203','EMT122','MAT002','FIS069','DCC204','DCC205','DCC218','MAT015','FIS086','FIS152','ELE064','ELT124','EST773','ELE065','ELE028','ELT029','ELT084','ELT123','ELE631','ESA019','ELT136','EEE050','ELE082','ELE077','EEE048']),
    "6. Avançado — Reta final":    ("fechar matérias finais do curso", ['MAT001','MAT038','QUI628','ELE630','DCC217','MAT039','FIS065','DCC203','EMT122','MAT002','FIS069','DCC204','DCC205','DCC218','MAT015','FIS086','FIS152','ELE064','ELT124','EST773','ELE065','ELE028','ELT029','ELT084','ELT123','ELE631','ESA019','ELT136','EEE050','ELE082','ELE077','ELE632','ELE633','EEE046','EEE017','ELE088','EEE048','EEE049']),
}

resultados = []
for nome, (interesse, concluidas) in cenarios.items():
    perfil = get_completed_profile_text(df, concluidas)
    q = f"{perfil} {interesse}"
    q_emb = encode_texts(model, [q], batch_size=1)[0]
    hits = semantic_search(q_emb, corpus_embeddings, top_k=20)
    cands = []
    for h in hits:
        r = df.iloc[h['index']].copy()
        r['predicted_probability'] = h['score']
        r['difficulty'] = r['dificuldade_geral']
        cands.append(r)
    df_c = pd.DataFrame(cands)
    df_c = df_c[~df_c['codigo'].isin(concluidas)]
    df_v = filter_recommended_candidates(df_c, concluidas)
    df_f = greedy_balance_by_difficulty(df_v, difficulty_budget=22.0, max_courses=5, respect_schedule=True)
    resultados.append({
        "Cenário": nome,
        "Top-20": len(df_c),
        "Pré-req OK": len(df_v),
        "Recomendadas": len(df_f),
        "Top-1": df_f.iloc[0]['codigo'] if len(df_f) > 0 else "—",
        "Top-1 Score": f"{df_f.iloc[0]['predicted_probability']:.3f}" if len(df_f) > 0 else "—",
        "Dificuldade total": f"{df_f['difficulty'].sum():.1f}" if len(df_f) > 0 else "—",
    })

display(pd.DataFrame(resultados).set_index("Cenário"))

---
## Verificação de Conflitos de Horário

Exemplo de como o filtro de horários evita choques: simulamos a seleção de disciplinas com e sem o filtro `respect_schedule`.

In [ ]:
# Aluno no 3º período — tenta montar grade com e sem filtro de horário
concluidas_sched = ['MAT001','MAT038','QUI628','ELE630','DCC217','MAT039','FIS065','DCC203','EMT122']
perfil = get_completed_profile_text(df, concluidas_sched)
q = f"{perfil} programação, computação, eletrônica"
q_emb = encode_texts(model, [q], batch_size=1)[0]
hits = semantic_search(q_emb, corpus_embeddings, top_k=20)

cands = []
for h in hits:
    r = df.iloc[h['index']].copy()
    r['predicted_probability'] = h['score']
    r['difficulty'] = r['dificuldade_geral']
    cands.append(r)
df_c = pd.DataFrame(cands)
df_c = df_c[~df_c['codigo'].isin(concluidas_sched)]
df_v = filter_recommended_candidates(df_c, concluidas_sched)

print("=== SEM filtro de horário ===")
df_sem = greedy_balance_by_difficulty(df_v, difficulty_budget=22.0, max_courses=5, respect_schedule=False)
for _, row in df_sem.iterrows():
    slots = get_schedule_slots(row)
    h_str = ', '.join(f'{d} {t}' for d,t in slots) if slots else '—'
    print(f"  {row['codigo']}  {h_str}")

# Detecta conflitos
all_slots = []
for _, row in df_sem.iterrows():
    all_slots.extend(get_schedule_slots(row))
from collections import Counter
conflicts = [(s, c) for s, c in Counter(all_slots).items() if c > 1]
print(f"\n  ⚠ Conflitos detectados: {len(conflicts)}")
for slot, count in conflicts:
    print(f"    {slot[0]} {slot[1]} — {count}x")

print("\n=== COM filtro de horário ===")
df_com = greedy_balance_by_difficulty(df_v, difficulty_budget=22.0, max_courses=5, respect_schedule=True)
for _, row in df_com.iterrows():
    slots = get_schedule_slots(row)
    h_str = ', '.join(f'{d} {t}' for d,t in slots) if slots else '—'
    print(f"  {row['codigo']}  {h_str}")

---
## Fine-Tuning do BERT Classificador (3 Estágios Progressivos)

Além da busca semântica com SBERT, o projeto treina um **classificador BERT** binário que decide se uma disciplina é adequada para o perfil do aluno (label 1 = relevante, 0 = irrelevante).

A estratégia é **Transfer Learning progressivo em 3 estágios**:
1. **Feature Extraction** — só treina a cabeça de classificação (encoder congelado)
2. **Partial Unfreeze** — descongela as 2 últimas camadas do encoder
3. **Full Fine-Tuning** — modelo inteiro treinado com learning rate baixo

Isso evita overfitting em datasets pequenos e aproveita o conhecimento prévio do BERTimbau.

In [ ]:
from src.synthetic_profile_generator import (
    generate_period_prefix_profiles,
    generate_elective_variants,
    generate_labelled_pairs,
    split_by_profile,
)
from src.baseline_no_text import train_baseline, evaluate_baseline
from src.evaluate import classification_metrics, compare_models

# 1. Gerar perfis sintéticos e pares de treino
base_profiles = generate_period_prefix_profiles(df)
all_profiles = base_profiles + generate_elective_variants(df, base_profiles)
pairs = generate_labelled_pairs(df, all_profiles)
splits = split_by_profile(pairs, seed=42)

print(f"Perfis sintéticos: {len(all_profiles)}")
print(f"Pares de treino:   {len(splits['train'])} (train)  |  {len(splits['validation'])} (val)  |  {len(splits['test'])} (test)")
print(f"Distribui\u00e7\u00e3o de labels (train):\n{splits['train']['label'].value_counts().to_dict()}")

In [ ]:
# 2. Baseline — Regressão Logística sobre features estruturais (sem texto)
print("Treinando baseline estrutural...")
baseline = train_baseline(splits['train'])
baseline_metrics = evaluate_baseline(baseline, splits['test'])
print(f"Baseline (Logistic Regression) \u2192 Accuracy: {baseline_metrics['accuracy']:.3f}  |  F1: {baseline_metrics['f1']:.3f}")

In [ ]:
# 3. Fine-tuning progressivo do BERT (3 estágios)
from src.train_bert_classifier import fit_three_stage_model

print("Iniciando fine-tuning do BERT (3 est\u00e1gios)...")
bert_model, history = fit_three_stage_model(splits['train'], splits['validation'], batch_size=4, seed=42)

print("\nHist\u00f3rico de treino:")
for entry in history:
    print(f"  {entry['stage']:22s} \u2192 loss: {entry['loss']:.4f}  acc: {entry['accuracy']:.3f}  f1: {entry['f1']:.3f}")

In [ ]:
# 4. Comparação final Baseline vs BERT
from src.train_bert_classifier import PairTextDataset, create_tokenizer, predict
from torch.utils.data import DataLoader

# Avaliar BERT no conjunto de teste
tokenizer = create_tokenizer()
test_dataset = PairTextDataset(splits['test'], tokenizer)
test_loader = DataLoader(test_dataset, batch_size=4)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_model.to(device)

y_true, y_pred, _ = predict(bert_model, test_loader, device)
bert_metrics = classification_metrics(y_true, y_pred)

comparison = compare_models({
    "Baseline (Log. Reg.)": baseline_metrics,
    "BERT (fine-tuned)": bert_metrics,
})
print("\nCompara\u00e7\u00e3o no conjunto de teste:")
display(comparison)